# Chapter 1 - EXTRACT 

#### Key insights 
1. ** 45 API(Application Programming Interface) Calls. 
2. ** 15 countries
3. ** 3 indicators
4. ** Rate limiting 
5. ** Error handling 
6. ** 675 records 

## 01 - CONFIGURATION AND API SETUP 

In [21]:
import requests       # "request": Para comunicarse con Web sites. 
import pandas as pd 
import json          #"JSON: JavaScript Object Notation. 

#World Bank API base URL 
BASE_URL = "http://api.worldbank.org/v2"


## 02 - FILTERING

In [22]:
#Indicators we want to extract (Relevant for credit risk) 

INDICATORS = {
    "NY.GDP.PCAP.CD": "gdp_per_capita",     # GDP per capita (current US$)
    "FP.CPI.TOTL.ZG": "inflation_rate",     # Inflation, consumer prices (annual %)
     "SL.UEM.TOTL.ZS": "unemployment_rate"  # Unemployment, total (% of labor force)
}

# Countries — mix от EU, emerging markets, developed
COUNTRIES = [
    "BGR",  # Bulgaria
    "ROU",  # Romania
    "DEU",  # Germany
    "GBR",  # United Kingdom
    "USA",  # United States
    "POL",  # Poland
    "HUN",  # Hungary
    "CZE",  # Czech Republic
    "GRC",  # Greece
    "ITA",  # Italy
    "ESP",  # Spain
    "FRA",  # France
    "TUR",  # Turkey
    "IRL",  # Ireland (Experian HQ!)
    "NLD"   # Netherlands
]# Date range: Los ultimos 15 años 
DATE_RANGE = "2011:2025"

print(f" Ready to extract {len (INDICATORS)} indicators for {len(COUNTRIES)} countries" )
print(f" Date range: {DATE_RANGE}")


 Ready to extract 3 indicators for 15 countries
 Date range: 2011:2025


## 03 - API CONNECTION AND DATA RETRIEVAL 

#### RECALL
- HTTP (Hypertext Transfer Protocol):
Es un protocolo que permite la comunicación entre tu programador y un servidor. 
response = requests.get(url)

- API Conection: 
En lugar de entrar manualmente a la página del Banco Mundial, buscar el dato y copiarlo, Python puede pedirlo automáticamente.

#### RECALL Step-by-Step Guide: Building a World Bank API Function
1. Create the function and the specify the parameters. 


2. Basic function structure:

   ```python
   def nombre_de_la_funcion(parametro1, parametro2):
       # Instrucciones
       return resultado
   ```
3. En "Comprobar el código HTTP" no olvidar. 
Terminar la función devolviendo una lista vacia cuando hay un error
```python
   return [] 
   ```
4. Un programador cuando trabaja con una API nueva: investiga cómo están organizados los datos antes de escribir el código, el Banco Mundial organiza la respuesta en metadatos. 

https://datahelpdesk.worldbank.org/knowledgebase/articles/898581-api-basic-call-structures


### Python List Indexing and Data Validation

| Código | Significado |
|---|---|
| `data[0]` | Primer elemento de la lista |
| `data[1]` | Segundo elemento de la lista |
| `len(data)` | Cantidad de elementos |
| `data[1] is None` | Comprueba si el segundo elemento es `None` |
| `return data[1]` | Devuelve el segundo elemento |


### Recall Functions - JSON Data Processing

| Function / Parameter | Description |
|---|---|
| `response.json()` | Interprets a JSON response and converts it into Python objects. |
| `json.dumps()` | Converts a Python object into a JSON-formatted string. |
| `indent=2` | Applies two spaces of indentation per level to make JSON easier to read. |



In [33]:
import requests
import json

# 1.- DEFINE: Function and parameters
def fetch_worldbank_data(country_code, indicator_code):
    """Fetch data from World Bank API for a specific country and indicator."""

    # BUILD: Construir la URL
    url = f"{BASE_URL}/country/{country_code}/indicator/{indicator_code}"

    # 2.- CONFIGURE: Configurar los parámetros
    params = {
        "format": "json",
        "date": DATE_RANGE,
        "per_page": 100                       #Registros por pagina. 
    }

    # 3.- REQUEST: Enviar la solicitud HTTP
    try:
        response = requests.get(url, params=params, timeout=60)

    except requests.exceptions.Timeout:
        print(f"Timeout for {country_code} - {indicator_code}")
        return []

    except requests.exceptions.RequestException as e:
        print(f"Request error for {country_code} - {indicator_code}: {e}")
        return []

    # 4.- CHECK: Comprobar el código HTTP
    if response.status_code != 200:
        print(f"Error for {country_code} - {indicator_code}: Status {response.status_code}")
        return []

    # 5.- PARSE: Interpretar la respuesta JSON
    try:
        data = response.json()

    except ValueError:
        print(f"Invalid JSON for {country_code} - {indicator_code}")
        return []

    # 6.- VALIDATE: Comprobar que existan registros económicos
    if not isinstance(data, list) or len(data) < 2 or not data[1]:
        print(f"No data for {country_code} - {indicator_code}")
        return []

    # 7.- RETURN: Devolver los registros económicos
    return data[1]


# TEST: Ejecutar una consulta de prueba
test_data = fetch_worldbank_data("BGR", "NY.GDP.PCAP.CD")


# INSPECT: Contar y mostrar registros
print(f"Got {len(test_data)} records for Bulgaria GDP per capita")
print("\nFirst record sample:")

if test_data:
    print(json.dumps(test_data[0], indent=5))
else:
    print("No records available to display.")

Got 15 records for Bulgaria GDP per capita

First record sample:
{
     "indicator": {
          "id": "NY.GDP.PCAP.CD",
          "value": "GDP per capita (current US$)"
     },
     "country": {
          "id": "BG",
          "value": "Bulgaria"
     },
     "countryiso3code": "BGR",
     "date": "2025",
     "value": 20328.1667065538,
     "unit": "",
     "obs_status": "",
     "decimal": 1
}


Observations: Diferencia entre JSON y objectos de PYTHON 

  ```JSON 
   {
  "country": "Bulgaria",
  "year": "2024",
  "gdp_per_capita": 17596.02
}
   ```
JSON es un formato para intercambiar información. Python necesita interpretarlo como sus propios objetos para manipularlo fácilmente.

  ```python
   data = {
    "country": "Bulgaria",
    "year": "2024",
    "gdp_per_capita": 17596.02
}
   ```

## 04- AUTOMATED DATA EXTRACTION 

### Recall Functions — Automated Data Extraction

4.-Crear la lista de diccionarios: Recuerda debe de llevar la siguiente estructura. 

```python
"nuevo_campo": transformacion(registro["campo_original"])
  
   ```
Aunque no siempre necesita una "Transformación". 
| Function | Description |
|---|---|
| `.append()` | Adds a new dictionary to `all_records`. |
| `record["country"]["value"]` | Extracts the country name from the JSON data. |
| `int(record["date"])` | Converts the year from a string (`"2024"`) to an integer (`2024`). |



### Recall Functions — Understanding `for` Loops and Records

#### 1. Difference Between `records` and `record`

| Variable | Description |
|---|---|
| `records` | A list containing all records retrieved from the API. |
| `record` | A single record from the list, processed during each loop iteration. |
| `for record in records:` | Iterates through the list and processes one record at a time. |
| `record["date"]` | Retrieves the date from the current record. |

#### 2. Practical Example

```python
records = [
    {"date": "2024", "value": 100},
    {"date": "2023", "value": 200},
    {"date": "2022", "value": 300}
]

for record in records:
    print(record["date"])
```

**Expected Output:**

```text
2024
2023
2022
```

#### 3. How the Loop Works

| Iteration | Current `record` | `record["date"]` |
|---|---|---|
| 1 | `{"date": "2024", "value": 100}` | `"2024"` |
| 2 | `{"date": "2023", "value": 200}` | `"2023"` |
| 3 | `{"date": "2022", "value": 300}` | `"2022"` |

In [34]:
import time
#1.- Prepara el contenedor: Crea una lista vacía
all_records = []
#2.- Recorre los paises and indicators
for country in COUNTRIES: 
    for indicator_code, indicator_name in INDICATORS.items():
        
        #3.-Fetch data:Llamar a la API automáticamente
        records = fetch_worldbank_data(country, indicator_code)
        
        #4.- Agrega datos y organizalos: CREAR LA LISTA DE DICCIONARIOS
        for record in records: 
            all_records.append({ 
                "country_code": country, 
                "country_name": record["country"]["value"], 
                "indicator_code":indicator_code, 
                "indicator_name":indicator_name, 
                "year": int(record["date"]),
                "value": record["value"]
            })
         #5.- Hacer que Python espere 0.1 segundos en cada interación donde aparece esta instrucción. 
        time.sleep(0.1) 
    
    #6.-Comprobar los resultados:
    print (f"✓ {country} completed")

print(f"\n✅ Extraction complete!")
print(f"Total records collected: {len(all_records)}")
print(f"\nSample record:")
print(json.dumps(all_records[0], indent=2))


✓ BGR completed
✓ ROU completed
✓ DEU completed
✓ GBR completed
✓ USA completed
✓ POL completed
✓ HUN completed
✓ CZE completed
✓ GRC completed
✓ ITA completed
✓ ESP completed
✓ FRA completed
✓ TUR completed
✓ IRL completed
✓ NLD completed

✅ Extraction complete!
Total records collected: 675

Sample record:
{
  "country_code": "BGR",
  "country_name": "Bulgaria",
  "indicator_code": "NY.GDP.PCAP.CD",
  "indicator_name": "gdp_per_capita",
  "year": 2025,
  "value": 20328.1667065538
}


## 05 - EXTRACTION RESULTS


### 05 - ¿Qué datos recibí y cómo están estructurados?

| # | Step | Objective | Common Functions |
|---|---|---|---|
| 1 | Create DataFrame | Convertir los registros extraídos en una tabla | `pd.DataFrame()` |
| 2 | Check Shape | Comprobar cuántas filas y columnas tenemos | `.shape`, `len()` |
| 3 | Inspect Columns | Identificar los campos disponibles | `.columns`, `.columns.tolist()` |
| 4 | Check Data Types | Revisar qué tipo de dato tiene cada columna | `.dtypes`, `.info()` |
| 5 | Preview Records | Visualizar los primeros registros | `.head()`, `.sample()` |


In [35]:
#1. Convertir los registros en un DataFrame
df= pd.DataFrame(all_records) 

#2. Comprobar el tamaño  print(df.shape)
print("=" * 50 )
print("DATAFRAME OVERVIEW") 
print( "=" *50)
print(f"\nShape:{df.shape}")

#3.Identificar las columnas print(df.columns.tolist()) 
print(f"\nColumn types:")
      
#4.Revisar los tipos de datos 
print(df.dtypes) 
      
#5.Visualizar los primeros registros 
print(f"\nFirst 5 rows:")
df.head()

DATAFRAME OVERVIEW

Shape:(675, 6)

Column types:
country_code       object
country_name       object
indicator_code     object
indicator_name     object
year                int64
value             float64
dtype: object

First 5 rows:


,country_code,country_name,indicator_code,indicator_name,year,value
0,BGR,Bulgaria,NY.GDP.PCAP.CD,gdp_per_capita,2025,20328.166707
1,BGR,Bulgaria,NY.GDP.PCAP.CD,gdp_per_capita,2024,17596.916762
2,BGR,Bulgaria,NY.GDP.PCAP.CD,gdp_per_capita,2023,15854.019289
3,BGR,Bulgaria,NY.GDP.PCAP.CD,gdp_per_capita,2022,13999.911336
4,BGR,Bulgaria,NY.GDP.PCAP.CD,gdp_per_capita,2021,12966.808520


### 05 - ¿Son correctos, completos y consistentes esos datos?
Ya se extrajeron 675 registros económicos de la API. Ahora necesitamos inspeccionarlos para segurar que Python los recibió y organizó como se espera. 

In [ ]:
#Data Quality Checks
print("="* 50) 
print("DATA QUALITY CHECKS")
print("="*50 )

#1.Check for missing values 
print("\n1.Missing values per column: ")
print(df.isnull().sum()) 

#2.Check for duplicates
duplicates = df.duplicated().sum() 
print(f"\n3.Duplciate rows: {duplicates}.")

#3.Year range validation
print(f"\n3.Year range: {df['year'].min()} - {df['year'].max()}")

#4.Value statistics per Indicator 
print("\n4.Value statistics per indicator:")
print(df.groupby('indicator_name')['value'].describe()[['count','mean','min','max']])

#5.Records per country 
print("\n5. Records per country:")
print(df['country_code'].value_counts())



DATA QUALITY CHECKS

1.Missing values per column: 
country_code      0
country_name      0
indicator_code    0
indicator_name    0
year              0
value             1
dtype: int64

3.Duplciate rows: 0.

3.Year range: 2011 - 2025

4.Value statistics per indicator:
                   count          mean          min            max
indicator_name                                                    
gdp_per_capita     225.0  34785.692307  7269.025865  131592.462548
inflation_rate     224.0      4.100662    -1.725612      72.308830
unemployment_rate  225.0      8.061173     2.015000      27.686000

5. Records per country:
country_code
BGR    45
ROU    45
DEU    45
GBR    45
USA    45
POL    45
HUN    45
CZE    45
GRC    45
ITA    45
ESP    45
FRA    45
TUR    45
IRL    45
NLD    45
Name: count, dtype: int64


### ¿Tus datos tienen coherencia y consistencia?
Los resultados parecen económicamente plausibles. 
En calidad de datos distinguimos varios conceptos:
- Completeness (completitud): ¿Faltan datos? Tienes un valor ausente.
- Uniqueness (unicidad): ¿Hay registros duplicados? No hay filas idénticas duplicadas.
- Validity (validez): ¿Los valores tienen sentido y cumplen las reglas? Una inflación negativa es posible.
- Consistency (consistencia): ¿Los datos son compatibles entre sí y respetan las mismas reglas?
- Accuracy (exactitud): ¿Coinciden con la información real de la fuente? Para comprobarlo, habría que contrastarlos con World Bank.

In [ ]:
#Checking what is the missing value
df[df["value"].isna()]

,country_code,country_name,indicator_code,indicator_name,year,value
195,USA,United States,FP.CPI.TOTL.ZG,inflation_rate,2025,NaN


In [ ]:
# TRANSFORM: Remove missing economic values
df_clean = df.dropna(subset=["value"]).copy()

# VALIDATE: Check results
print(f"Original records: {len(df)}")
print(f"Clean records: {len(df_clean)}")
print(f"Missing values: {df_clean['value'].isna().sum()}")

Original records: 675
Clean records: 674
Missing values: 0


## 06 - SAVE EXTRACTED DATA  

In [ ]:
from pathlib import Path

# Folder where the data will be stored
data_dir = Path("../data")
data_dir.mkdir(parents=True, exist_ok=True)

# Save cleaned data
df_clean.to_csv(
    data_dir / "worldbank_long.csv",
    index=False
)

print("Chapter 1 completed!")
print(f"Records saved: {len(df_clean)}")

Chapter 1 completed!
Records saved: 674
